In [2]:
from IPython.core.magic import register_cell_magic, register_line_magic
import subprocess
import tempfile
import os
import re
import shutil

_JAVA_COMPILE_DIR = None
_JAVA_CLASSPATH = []


def _get_compile_dir():
    global _JAVA_COMPILE_DIR
    if _JAVA_COMPILE_DIR is None:
        _JAVA_COMPILE_DIR = tempfile.mkdtemp(prefix="java_magic_")
    return _JAVA_COMPILE_DIR


def _cleanup():
    global _JAVA_COMPILE_DIR, _JAVA_CLASSPATH
    if _JAVA_COMPILE_DIR and os.path.exists(_JAVA_COMPILE_DIR):
        shutil.rmtree(_JAVA_COMPILE_DIR, ignore_errors=True)
    _JAVA_COMPILE_DIR = None
    _JAVA_CLASSPATH = []


@register_line_magic
def javacp(line):
    global _JAVA_CLASSPATH
    path = line.strip().strip('"').strip("'")
    if os.path.exists(path):
        _JAVA_CLASSPATH.append(path)
        print("[OK] Added to classpath: " + path)
    else:
        print("[ERR] Not found: " + path)


@register_line_magic
def javacls(line):
    _cleanup()
    print("[OK] Java compile cache cleared")


@register_cell_magic
def java(line, cell):
    opts = []
    parts = line.strip().split()
    while parts and parts[0].startswith('--'):
        opts.append(parts.pop(0))
    
    force_new = '--new' in opts
    show_time = '--time' in opts
    
    if force_new:
        _cleanup()
    
    compile_dir = _get_compile_dir()
    
    public_class = re.search(r'public\s+class\s+(\w+)', cell)
    package_class = re.search(r'class\s+(\w+)', cell)
    
    if public_class:
        classname = public_class.group(1)
    elif package_class:
        classname = package_class.group(1)
    else:
        classname = "Main"
    
    if parts and parts[0][0].isupper() and not parts[0].isdigit():
        cmd_classname = parts.pop(0)
        if cmd_classname != classname and not public_class and not package_class:
            classname = cmd_classname
    else:
        cmd_classname = classname
    
    args = parts
    
    package_match = re.search(r'package\s+([\w.]+);', cell)
    if package_match:
        package_path = package_match.group(1).replace('.', os.sep)
        src_dir = os.path.join(compile_dir, package_path)
        os.makedirs(src_dir, exist_ok=True)
    else:
        src_dir = compile_dir
    
    filepath = os.path.join(src_dir, classname + ".java")
    
    code = cell
    if not re.search(r'class\s+\w+', cell):
        code = (
            "public class " + classname + " {\n"
            "    public static void main(String[] args) {\n"
            "        " + cell.replace('\n', '\n        ') + "\n"
            "    }\n"
            "}"
        )
    
    has_scanner = 'Scanner' in cell and 'System.in' in cell
    input_data = None
    
    if has_scanner and args:
        input_data = '\n'.join(args) + '\n'
        args = []
    
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(code)
    
    javac_cmd = ['javac', '-encoding', 'UTF-8']
    if _JAVA_CLASSPATH:
        cp = os.pathsep.join(_JAVA_CLASSPATH)
        javac_cmd.extend(['-cp', cp])
    javac_cmd.append(filepath)
    
    compile_result = subprocess.run(
        javac_cmd,
        capture_output=True,
        text=True,
        cwd=compile_dir
    )
    
    if compile_result.returncode != 0:
        errors = compile_result.stderr
        errors = errors.replace(compile_dir + os.sep, '')
        errors = errors.replace(compile_dir, '')
        print("[COMPILE ERROR]\n")
        print(errors)
        return
    
    java_cmd = ['java', '-Dfile.encoding=UTF-8']
    if _JAVA_CLASSPATH:
        cp = os.pathsep.join([compile_dir] + _JAVA_CLASSPATH)
        java_cmd.extend(['-cp', cp])
    else:
        java_cmd.extend(['-cp', compile_dir])
    
    if package_match:
        full_classname = package_match.group(1) + "." + cmd_classname
    else:
        full_classname = cmd_classname
    
    java_cmd.append(full_classname)
    java_cmd.extend(args)
    
    import time
    start = time.time()
    
    try:
        if input_data:
            run_result = subprocess.run(
                java_cmd,
                capture_output=True,
                text=True,
                cwd=compile_dir,
                input=input_data,
                timeout=10
            )
        else:
            run_result = subprocess.run(
                java_cmd,
                capture_output=True,
                text=True,
                cwd=compile_dir,
                timeout=10
            )
    except subprocess.TimeoutExpired:
        elapsed = time.time() - start
        print("[TIMEOUT] Execution timed out after " + str(round(elapsed, 2)) + "s")
        if has_scanner and not input_data:
            print("\n[HINT] Scanner(System.in) detected but no input provided")
            print("       Solutions:")
            print("       1. Hardcode input values")
            print("       2. Pass input as args: %%java ClassName 20")
            print("       3. Use file redirection")
        return
    
    elapsed = time.time() - start
    
    if run_result.stdout:
        print(run_result.stdout, end='')
    
    if run_result.stderr:
        stderr = run_result.stderr
        harmless = ['Picked up JAVA_TOOL_OPTIONS', 'WARNING']
        if not any(h in stderr for h in harmless):
            print("\n[STDERR] " + stderr.strip())
    
    if show_time:
        print("\n[TIME] " + str(round(elapsed, 3)) + "s")


del java, javacp, javacls

In [5]:
%%java Car

public class Car {
    private String brand;
    private String color;

    public Car() {
        System.out.println("无参构造.......");
    }

    public Car(String brand, String color) {
        this.brand = brand;
        this.color = color;
        System.out.println("有参构造.......");
    }

    public void show() {
        System.out.println("品牌是：" + brand + ",颜色是：" + color);
    }

    public String getBrand() {
        return brand;
    }

    public String getColor() {
        return color;
    }

    public void setBrand(String brand) {
        this.brand = brand;
    }

    public void setColor(String color) {
        this.color = color;
    }

    // 占位 main，防止报错
    public static void main(String[] args) {}
}

In [6]:
%%java Test01

public class Test01 {
    public static void main(String[] args) {
        Car car1 = new Car();
        car1.show();

        Car car2 = new Car("奔驰", "红色");
        car2.show();
    }
}

无参构造.......
品牌是：null,颜色是：null
有参构造.......
品牌是：奔驰,颜色是：红色


In [7]:
%%java User

public class User {
    private String name;

    public User(String name) {
        this.name = name;
    }

    public User() {
        this("默认用户");
    }

    public void showName() {
        System.out.println("用户名是：" + this.name);
    }

    public void test() {
        this.showName();
    }

    // 占位 main
    public static void main(String[] args) {}
}

In [8]:
%%java Test02

public class Test02 {
    public static void main(String[] args) {
        User user1 = new User();
        user1.test();

        User user2 = new User("张三");
        user2.showName();
    }
}

用户名是：默认用户
用户名是：张三


In [9]:
%%java Emp

public class Emp {
    private String name;
    private double salary;

    public Emp(String name, double salary) {
        this.name = name;
        this.salary = salary;
    }

    public void info() {
        System.out.println("员工:" + this.name + ",薪水:" + this.salary);
    }

    public static void main(String[] args) {}
}

In [10]:
%%java Test03

public class Test03 {
    public static void main(String[] args) {
        Emp emp1 = new Emp("张三", 5000);
        emp1.info();
    }
}

员工:张三,薪水:5000.0


In [11]:
%%java Animal

public class Animal {
    public String name;

    public void eat() {
        System.out.println(name + "吃东西");
    }

    public static void main(String[] args) {}
}

In [12]:
%%java Dog

public class Dog extends Animal {
    public void bark() {
        System.out.println(name + "汪汪叫");
    }

    public static void main(String[] args) {}
}

In [13]:
%%java Test04

public class Test04 {
    public static void main(String[] args) {
        Dog dog = new Dog();
        dog.name = "旺财";
        dog.eat();
        dog.bark();
    }
}

旺财吃东西
旺财汪汪叫


In [14]:
%%java Person

public class Person {
    protected String name;

    public void run() {
        System.out.println(name + "在跑步");
    }

    public static void main(String[] args) {}
}

In [15]:
%%java Student

public class Student extends Person {
    public void study() {
        System.out.println(name + "正在学习");
    }

    public static void main(String[] args) {}
}

In [16]:
%%java Test05

public class Test05 {
    public static void main(String[] args) {
        Student s = new Student();
        s.name = "张三";
        s.run();
        s.study();
    }
}

张三在跑步
张三正在学习


In [17]:
%%java Father

public class Father {
    String name = "父类";

    public Father(String name) {
        this.name = name;
        System.out.println("父类构造:" + name);
    }

    public void show() {
        System.out.println("父类show");
    }

    public static void main(String[] args) {}
}

In [18]:
%%java Son

public class Son extends Father {
    String name = "子类";

    public Son() {
        super("父类");
    }

    public void print() {
        System.out.println(super.name);
        super.show();
    }

    public static void main(String[] args) {}
}

In [19]:
%%java Test05a

public class Test05a {
    public static void main(String[] args) {
        Son son = new Son();
        System.out.println("子类name:" + son.name);
        son.print();
    }
}

父类构造:父类
子类name:子类
父类
父类show


In [20]:
%%java Animal2

public class Animal2 {
    String type = "动物";

    public Animal2(String type) {
        System.out.println("父类:" + type);
    }

    public static void main(String[] args) {}
}

In [21]:
%%java Cat

public class Cat extends Animal2 {
    String type = "小猫";

    public Cat() {
        super("哺乳类");
    }

    public void test() {
        System.out.println("父类type:" + super.type);
        System.out.println("子类type:" + this.type);
    }

    public static void main(String[] args) {}
}

In [22]:
%%java Test07

public class Test07 {
    public static void main(String[] args) {
        Cat c = new Cat();
        c.test();
    }
}

父类:哺乳类
父类type:动物
子类type:小猫


In [23]:
%%java Animal3

public class Animal3 {
    public String name;

    public void eat() {
        System.out.println(name + "吃东西");
    }

    public void shout() {
        System.out.println("动物发声");
    }

    public static void main(String[] args) {}
}

In [24]:
%%java Dog

public class Dog extends Animal3 {
    public void bark() {
        System.out.println(name + "汪汪叫");
    }

    @Override
    public void shout() {
        System.out.println("小狗汪汪叫");
    }

    public static void main(String[] args) {}
}

In [25]:
%%java Test08

public class Test08 {
    public static void main(String[] args) {
        Animal3 a = new Animal3();
        a.shout();

        Dog dog = new Dog();
        dog.shout();
    }
}

动物发声
小狗汪汪叫


In [26]:
%%java Phone

package com.huaishida.test0708.pro2;

public class Phone {
    public void call() {
        System.out.println("普通手机打电话");
    }

    public static void main(String[] args) {}
}

In [27]:
%%java IPhone

package com.huaishida.test0708.pro2;

public class IPhone extends Phone {
    @Override
    public void call() {
        System.out.println("iPhone打电话");
    }

    public static void main(String[] args) {}
}

In [28]:
%%java Test1

package com.huaishida.test0708.pro2;

public class Test1 {
    public static void main(String[] args) {
        IPhone iPhone = new IPhone();
        iPhone.call();
    }
}

iPhone打电话


In [29]:
%%java Animal4

package com.huaishida.test0708.pro2;

public class Animal4 {
    public void shout() {
        System.out.println("动物发声");
    }

    public static void main(String[] args) {}
}

In [30]:
%%java Cat

package com.huaishida.test0708.pro2;

public class Cat extends Animal4 {
    @Override
    public void shout() {
        System.out.println("小猫喵喵叫");
    }

    public static void main(String[] args) {}
}

In [31]:
%%java Dog

package com.huaishida.test0708.pro2;

public class Dog extends Animal4 {
    @Override
    public void shout() {
        System.out.println("小狗汪汪叫");
    }

    public static void main(String[] args) {}
}

In [32]:
%%java Test22

package com.huaishida.test0708.pro2;

public class Test22 {
    public static void main(String[] args) {
        Animal4 a1 = new Cat();
        Animal4 a2 = new Dog();

        a1.shout();
        a2.shout();
    }
}

小猫喵喵叫
小狗汪汪叫


In [3]:
%%java Shape

package com.huaishida.test0708.pro2;

public class Shape {
    public double getArea() {
        System.out.println("计算图形的面积");
        return 0.0;
    }

    public static void main(String[] args) {}
}

In [4]:
%%java Circle

package com.huaishida.test0708.pro2;

public class Circle extends Shape {
    private double r;

    public Circle(double r) {
        this.r = r;
    }

    @Override
    public double getArea() {
        return 3.14 * r * r;
    }

    public static void main(String[] args) {}
}

In [5]:
%%java Rect

package com.huaishida.test0708.pro2;

public class Rect extends Shape {
    private double w, h;

    public Rect(double w, double h) {
        this.w = w;
        this.h = h;
    }

    @Override
    public double getArea() {
        return w * h;
    }

    public static void main(String[] args) {}
}

In [6]:
%%java Test3

package com.huaishida.test0708.pro2;

public class Test3 {
    public static void main(String[] args) {
        Rect rect = new Rect(10, 20);
        System.out.println("矩形的面积是：" + rect.getArea());

        Circle circle = new Circle(10);
        System.out.println("圆形的面积是：" + circle.getArea());
    }
}

矩形的面积是：200.0
圆形的面积是：314.0
